# Diffusion Distance Based Clustering

Execution notebook using the reusable functions in `ddbc_functions.py`.

# Import

In [ ]:
%load_ext autoreload
%autoreload 2

import gc
import os
import pickle
import matplotlib.pyplot as plt
import numpy as np
from datetime import datetime

from ddbc_functions import (
    DDBCConfig,
    big_objects,
    build_column_aggregation_matrix,
    build_interlayer_coupling_matrices,
    build_interlayer_coupling_matrices_randomized,
    randomize_layer_adjacency_matrix,
    build_kNN,
    build_layer_adjacency_matrices,
    build_multilayer_transition_matrix,
    build_row_aggregation_matrix,
    build_distinct_gene_mapping,
    compute_and_save_diffusion_distance,
    compute_and_save_average_transition_matrix,
    compute_stationary_layer_weights,
    load_bootstrap_scores,
    load_diffusion_distance,
    load_matrices,
    plot_stationary_layer_weights,
    run_bootstrap_robustness,
    run_clustering,
    select_communities,
    stationary_distribution,
    is_symmetric,
)

In [ ]:
run_id = datetime.now().strftime("%Y%m%d_%H%M%S")

# Configuration

In [ ]:
# Bipolar is defaulted to be the starting point of randomization. All NULL folders are copies of BIPOLAR folders.
DISEASE = "NULL"

config = DDBCConfig(
    disease=DISEASE,
    average_t=[2, 4, 6, 8],
    interlayer_transition_prob=0.35,
    num_neighbors=400,
    resolution=1.3,
    size_cap=100,
    score_cap=0,
    n_boots=1000,
    check_every=50,
    tol=0.01,
    patience=2,
    sampling_pct=0.8,
    leiden_seed=42,
)

# Building Multilayer Transition Matrix

In [ ]:
os.makedirs(config.output_directory, exist_ok=True)
os.makedirs(config.graph_directory, exist_ok=True)

matrices = load_matrices(config)
DGIDB_adjacency_matrix, MSIGDB_adjacency_matrix = (
    build_layer_adjacency_matrices(matrices)
)

## Coupling matrices and distinct gene indices

In [ ]:
NULL_MODEL = "random_dgidb"

In [ ]:
if (NULL_MODEL == "random_interconnectivity"):
    coupling = build_interlayer_coupling_matrices_randomized(
        DGIDB_adjacency_matrix,
        MSIGDB_adjacency_matrix,
        config,
        none_prob = 0.015
    )
elif NULL_MODEL == "random_dgidb":
    # Rewire the DGIDB layer itself and keep the real gene-to-gene coupling.
    # seed is left unset so each run draws a fresh null realization.
    DGIDB_adjacency_matrix = randomize_layer_adjacency_matrix(
        DGIDB_adjacency_matrix,
        swaps_per_edge = 10,
    )
    coupling = build_interlayer_coupling_matrices(
        DGIDB_adjacency_matrix,
        MSIGDB_adjacency_matrix,
        config,
    )
else:
    raise ValueError(f"Unknown NULL_MODEL: {NULL_MODEL}")
    

gene_to_index_distinct = build_distinct_gene_mapping(coupling, config)

## Construct the multilayer transition matrix

In [ ]:
P = build_multilayer_transition_matrix(
    DGIDB_adjacency_matrix,
    MSIGDB_adjacency_matrix,
    coupling,
)
num_genes = P.shape[0]

del DGIDB_adjacency_matrix, MSIGDB_adjacency_matrix
for _ in range(3):
    gc.collect()

## Stationary distribution

In [ ]:
pi = stationary_distribution(
    P,
    tol=config.stationary_tol,
    maxit=config.stationary_maxit,
    seed=config.stationary_seed,
)
pi

# Aggregation

## Column aggregation

In [ ]:
A_c = build_column_aggregation_matrix(
    P,
    coupling,
    gene_to_index_distinct,
)

## Preparing weights

In [ ]:
wD_list, wM_list = compute_stationary_layer_weights(pi, coupling)
plot_stationary_layer_weights(wD_list, wM_list, config, run_id = run_id)

## Row aggregation

In [ ]:
A_r, wD_list2, wM_list2 = build_row_aggregation_matrix(
    P,
    pi,
    coupling,
    gene_to_index_distinct,
)

assert wD_list == wD_list2 and wM_list == wM_list2

# Matrix-Free Method

In [ ]:
P_t_agg_avg = compute_and_save_average_transition_matrix(
    P,
    config,
    A_r,
    A_c,
    dgidb_rows_only = True,
    run_id = run_id
)

big_objects()